# 08 · Tries

> Un albero in cui ogni arco è un carattere: le parole con lo stesso prefisso condividono il percorso. Cerchi un prefisso in `O(lunghezza)`, indipendentemente da quante parole hai.
> Struttura: [Big-O › Trie](../Fundamentals/Big-O%20e%20strutture%20dati.md#trie) · Negli esercizi: [Alberi e grafi › Trie](../Fundamentals/Alberi%20e%20grafi.md#trie-negli-esercizi)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Implement Trie (Prefix Tree) | nodo = figli + flag di fine parola | `class TrieNode`, `dict`, `setdefault` |
| Design Add and Search Words Data Structure | `.` = qualsiasi carattere → DFS su tutti i figli | ricorsione con indice, `any(...)` |
| Word Search II | trie delle parole + DFS sulla griglia; scendi nel trie mentre scendi nella griglia | backtracking in-place, rimuovere le parole trovate |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| figlio, creandolo se manca | `node = node.children.setdefault(ch, TrieNode())` |
| figlio, senza crearlo | `node = node.children.get(ch)` → `None` se manca |
| iterare i figli | `for ch, child in node.children.items()` |
| "almeno uno vero" | `any(dfs(child, i + 1) for child in node.children.values())` |
| figli in array fisso (solo `a`–`z`) | `[None] * 26`, indice `ord(ch) - ord("a")` |

## Il nodo come classe

Ogni nodo ha un dict `children` (carattere → nodo) e un flag `is_word`. Inserire = scendere creando i nodi mancanti; cercare = scendere fermandosi se un nodo manca.

`__slots__` (facoltativo) riduce la memoria per nodo, utile con decine di migliaia di parole.

In [1]:
class TrieNode:
    __slots__ = ("children", "is_word")

    def __init__(self) -> None:
        self.children: dict[str, "TrieNode"] = {}
        self.is_word = False

root = TrieNode()
for word in ("mug", "mugs", "cap"):
    node = root
    for ch in word:
        node = node.children.setdefault(ch, TrieNode())  # crea solo se manca
    node.is_word = True

print(sorted(root.children))                   # -> ['c', 'm']
print(root.children["m"].children["u"].children["g"].is_word)  # -> True

['c', 'm']
True


## Scendere seguendo un prefisso

Una funzione che restituisce il nodo raggiunto (o `None`) serve sia a `search` (poi controlli `is_word`) sia a `starts_with` (basta che il nodo esista): scrivila una volta.

In [2]:
# continua dall'esempio precedente
def walk(root: TrieNode, prefix: str) -> TrieNode | None:
    node = root
    for ch in prefix:
        node = node.children.get(ch)
        if node is None:
            return None
    return node

end = walk(root, "mu")
print(end is not None, end.is_word)        # -> True False: prefisso sì, parola no
print(walk(root, "mug").is_word, walk(root, "x"))  # -> True None

True False
True None


## Variante con dict annidati

Più compatta, comoda per Word Search II: ogni nodo è un dict, e una chiave speciale (`"$"`) contiene la parola completa che finisce lì. Così, quando la DFS la trova, hai già la parola senza ricostruirla.

In [3]:
END = "$"
trie: dict = {}
for word in ("eat", "ear", "oath"):
    node = trie
    for ch in word:
        node = node.setdefault(ch, {})
    node[END] = word

print(trie["e"]["a"].keys())  # -> dict_keys(['t', 'r'])
print(trie["e"]["a"]["t"])    # -> {'$': 'eat'}

dict_keys(['t', 'r'])
{'$': 'eat'}


## Wildcard: DFS su tutti i figli

Quando un carattere può essere "qualsiasi", non c'è un solo figlio da seguire: provi ciascuno e restituisci `True` se almeno uno porta a una parola. La DFS riceve il nodo e l'indice nella parola.

In [4]:
# continua dalla prima cella (TrieNode con mug, mugs, cap)
def matches(node: TrieNode, pattern: str, i: int = 0) -> bool:
    if i == len(pattern):
        return node.is_word
    ch = pattern[i]
    if ch == ".":
        return any(matches(child, pattern, i + 1) for child in node.children.values())
    child = node.children.get(ch)
    return child is not None and matches(child, pattern, i + 1)

print(matches(root, "m.g"), matches(root, "..p"), matches(root, "m..."), matches(root, "c."))  # -> True True True False

True True True False


## Trie + griglia (Word Search II)

L'idea: una sola DFS dalla griglia per **tutte** le parole insieme. Passi alla DFS il nodo del trie corrispondente al percorso; se la lettera della cella non è tra i figli, il ramo è morto e ti fermi subito.

Mattoni che ti servono:
- direzioni come lista di tuple e controllo dei bordi (vedi [Graphs](11%20-%20Graphs.ipynb));
- marcare la cella come visitata **in-place** (`board[r][c] = "#"`) e ripristinarla dopo la ricorsione;
- quando trovi una parola, `node.pop("$")` evita di trovarla due volte;
- potatura: se un nodo resta senza figli, rimuovilo dal padre, così i rami già esauriti non vengono riesplorati.

In [5]:
board = [list("oa"), list("et")]
r, c = 0, 1
saved = board[r][c]
board[r][c] = "#"       # visitata
# ... ricorsione sui vicini ...
board[r][c] = saved     # ripristino (backtracking)
print(board)            # -> [['o', 'a'], ['e', 't']]

node = {"t": {"$": "eat"}}
found = node["t"].pop("$", None)
print(found, node)      # -> eat {'t': {}}
if not node["t"]:
    del node["t"]       # potatura del ramo vuoto
print(node)             # -> {}

[['o', 'a'], ['e', 't']]
eat {'t': {}}
{}


## Errori comuni

- Confondere "il prefisso esiste" con "la parola esiste": serve `is_word`.
- Creare nodi durante la **ricerca** (usando `setdefault` invece di `get`): il trie si riempie di rami falsi.
- Word Search II senza rimuovere le parole trovate: duplicati nel risultato.
- Ripristinare la cella **dopo** ogni ritorno anticipato, non solo nel percorso normale.